# 6. ANÁLISES E INSIGHTS

Estruturadas as camadas do pipeline de dados, retomamos as perguntas inicialmente apresentadas para serem respondidas por meio da exploração dos dados:

> Após a edição da Resolução CNJ n.º 547, houve redução das execuções fiscais na Justiça Estadual de São Paulo?
> 
> Qual o perfil de distribuição das execuções fiscais entre os órgãos do Tribunal de Justiça do Estado de São Paulo?
> 
> Eventual redução decorrente da aplicação da Resolução CNJ n.º 547 ocorreu de maneira uniforme entre os diversos órgãos?

## 6.1. Quantidade de execuções no período

> Após a edição da Resolução CNJ n.º 547, houve redução das execuções fiscais na Justiça Estadual de São Paulo?

A primeira pergunta envolve a compreensão do quantitativo de execuções fiscais ao longo do período e se após a edição da Resolução CNJ n.º 547/2024, publicada em 22 de fevereiro de 2024, houve alteração desse volume.

Para esse objetivo, será realizada consulta na camada Gold da quantidade de processos, agrupando-se o resultado pelo ano e mês do ajuizamento. O código SQL da consulta segue abaixo:

In [0]:
%sql
SELECT gold_Tempo.ano_mes_ajuizamento, COUNT (numero_processo)
FROM gold_fatoajuizamento
JOIN gold_tempo on gold_fatoajuizamento.id_data = Gold_tempo.id_data
GROUP BY Gold_Tempo.ano_mes_ajuizamento
ORDER BY ano_mes_ajuizamento;
    


Databricks visualization. Run in Databricks to view.


![image_1789943178023.png](./image_1789943178023.png "image_1789943178023.png")


O resultado da consulta demonstra que houve uma diminuição bastante significativa da quantidade de ajuizamentos na proximidade da publicação da Resolução CNJ n.º 547/2024. Esse comportamento é explicado pelo fato de que o Tema n.º 1184 do STF, que trazia as conclusões que estão na regulamentação, já havia sido julgado, de forma que os entes públicos já tinham iniciado a adotar medidas para se adequar ao entendimento jurisprudencial acerca do ajuizamento.

Após a publicação, houve uma queda significativa do número de ajuizamentos, os quais permanecem atualmente em volume médio significativamente inferior àquele observado em 2023.

Interessante notar a existência de picos nos meses de dezembro ao longo dos anos, mesmo em 2023.  Essa sazonalidade pode estar relacionada a algum tipo de cobrança cuja exigibilidade recai nesse período, sendo um ponto que pode ser aprofundado em uma análise futura.


## 6.2. Distribuição das execuções por órgão judicial

> Qual o perfil de distribuição das execuções fiscais entre os órgãos do Tribunal de Justiça do Estado de São Paulo?

O segundo questionamento destina-se a compreender qual o perfil de distribuição das execuções fiscais entre os diversos órgãos judiciários do Tribunal de Justiça do Estado de São Paulo.

Para essa segunda análise, será realizada consulta na camada Gold da quantidade de processos, agrupando-se o resultado pelo órgão julgador. O código SQL dessa primeira consulta segue abaixo:



In [0]:
%sql
SELECT 
    gold_orgaojulgador.nome, 
    count(numero_processo) As Total_Execucoes
FROM gold_fatoajuizamento
JOIN gold_orgaojulgador on gold_fatoajuizamento.id_orgao = gold_orgaojulgador.id_orgao
GROUP BY gold_orgaojulgador.nome
ORDER BY total_execucoes DESC;

Os resultados dessa primeira consulta evidenciam que há uma concentração significativa de distribuição de execuções em determinadas unidades do Judiciário:

![image_1789944095736.png](./image_1789944095736.png "image_1789944095736.png")


Considerando a quantidade de registros e para possibilitar uma melhor análise, será inserido um recorte, por meio da inserção da cláusula LIMIT, para obtenção das 20 unidades judiciárias com maior número de execuções.

In [0]:
%sql
SELECT 
    gold_orgaojulgador.nome, 
    count(numero_processo) As Total_Execucoes
FROM gold_fatoajuizamento
JOIN gold_orgaojulgador on gold_fatoajuizamento.id_orgao = gold_orgaojulgador.id_orgao
GROUP BY gold_orgaojulgador.nome
ORDER BY total_execucoes DESC
LIMIT 20;

Databricks visualization. Run in Databricks to view.

![image_1789944377681.png](./image_1789944377681.png "image_1789944377681.png")

Extraindo-se os 20 primeiros registros, observa-se que o volume de distribuições é significativo superior à média na Vara de Execuções Fiscais Municipais da Capital (217.786 execuções). A Vara de Execuções Fiscais Estaduais da Capital ocupa a 4ª posição, com um quantitativo significativamente inferior de execuções ajuizadas (59.265).

Um outro achado relevante na análise é que o quantitativo de distribuição não segue a proporcionalidade que se poderia esperar do tamanho dos municípios em que sediadas as unidades judiciárias. Com efeito, o Setor de Execuções Fiscais de Campinas, que é o 3º maior município de São Paulo segundo o IBGE, ocupa a 11ª posição, havendo varas em municípios de menor porte com maior volume de ajuizamentos.



> Eventual redução decorrente da aplicação da Resolução CNJ n.º 547 ocorreu de maneira uniforme entre os diversos órgãos?

O terceiro questionamento destina-se a compreender como os efeitos da Resolução CNJ n.º 547 repercurtiram nos diversos órgãos judiciários do Tribunal de Justiça paulista.

Considerando o número de órgãos identificados, optou-se por realizar um recorte de um universo que possua maior representatividade estatística dentro do panorama analisado. Dessa maneira, a análise temporal tomou como base o grupo identificado na pergunta anterior, qual seja, os 20 órgãos judiciários com maior distribuição.

Para direcionar a consulta a este objetivo, foi utilizada uma CTE inicial na query, aproveitando o código utilizado na pergunta anterior, para possibilitar que o relacionamento com a tabela-fato Ajuizamento e a tabela-dimensão Tempo se limitassem ao grupo de órgãos que será analisado.




In [0]:
%sql
WITH Maiores_Unidades AS
    (SELECT 
        gold_orgaojulgador.nome,
        gold_orgaojulgador.id_orgao,
        count(numero_processo) As Total_Execucoes
    FROM gold_fatoajuizamento
    JOIN gold_orgaojulgador on gold_fatoajuizamento.id_orgao = gold_orgaojulgador.id_orgao
    GROUP BY gold_orgaojulgador.nome, gold_orgaojulgador.id_orgao
    ORDER BY total_execucoes DESC
    LIMIT 20)

SELECT 
    Maiores_Unidades.nome,
    gold_tempo.ano_mes_ajuizamento,
    count(numero_processo) As Total_Execucoes
FROM gold_fatoajuizamento
JOIN Maiores_Unidades on gold_fatoajuizamento.id_orgao = Maiores_Unidades.id_orgao
JOIN gold_tempo on gold_fatoajuizamento.id_data = Gold_tempo.id_data
GROUP BY Maiores_Unidades.nome, Gold_Tempo.ano_mes_ajuizamento
ORDER BY Gold_Tempo.ano_mes_ajuizamento;

Databricks visualization. Run in Databricks to view.

![image_1790452509392.png](./image_1790452509392.png "image_1790452509392.png")


O resultado da consulta traz uma demonstração que, no período imediatamente posterior à edição da Resolução CNJ n.º 547/2024, houve uma queda significiativa no número de ajuizamentos. Essa tendência foi seguida por uma posterior elevação, na maioria dos órgãos observados, seguido por uma estabilização do crescimento em momento posteriormente. Embora haja movimentos sazonais de alta dentro da série temporal, infere-se que, no geral, os patamares totais mantiveram-se inferiores àqueles observados antes da edição da normativa do Conselho Nacional de Justiça. Uma situação particular envolve a Vara de Execuções Fiscais Estaduais da Capital, que será tratada adiante.



![image_1790454557573.png](./image_1790454557573.png "image_1790454557573.png")




Diferentemente das demais unidades judiciárias, a Vara de Execuções Fiscais Estaduais da Capital - VEFE passou por uma variação diversa em virtude da alteração de sua competência no ano de 2025. Naquele ano, o Tribunal de Justiça paulista editou ato prevendo que a VEFE passaria a assumir a competência para processamento de todas as execuções fiscais da Capital e interior, com exceção daquelas que fossem ajuizadas no Núcleo 4.0 de Execuções Fiscais Estaduais.

Por esse motivo, observa-se que, após o movimento de baixa decorrente da vigência da Resolução CNJ n.º 547/2024, a VEFE experimentou um aumento do número de processos ajuizados, mantendo-se em um patamar inclusive superior ao observado anteriormente ao ato normativo.

Importante destacar que esse órgão atua exclusivamente com execuções fiscais propostas pelo Estado de São Paulo, o qual já adotava um piso de ajuizamento de 1200 UFESPs (Resolução PGE n.º 27/2017, a qual foi sucedida pela PGE n.º 9/2024) e também tinha como política, como regra, o encaminhamento dos débitos inscritos a protesto, de maneira que as condições impostas pela Resolução CNJ n.º 547/2024, por si só, tiveram menor repercussão no número de ajuizamentos realizados em âmbito estadual.


![image_1790455564780.png](./image_1790455564780.png "image_1790455564780.png")

Também importante contextualizar a situação do Núcleo Especializado de Justiça 4.0 - Execuções Fiscais Estaduais. 

O referido Núcleo foi criado em 2025 estaria destinado ao processamento de ações de maior valor ou que tivessem alguma relevância estratégica para o Estado de São Paulo. As ações com data de ajuizamento anterior a sua criação e que estão vinculadas a ele referem-se aos processos que tramitavam nas Varas de origem e foram redistribuídos ao Nucleo.

Dentro desse contexto, a série temporal evidencia que, após criação e implantação do Núcleo, o número de ajuizamentos mantém-se em um patamar mais reduzido e a maior parte dos casos associados ao órgão são de processos redistribuídos e ajuizados antes de sua implantação.

